In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../..").resolve()))

import ast
import pandas as pd
from src.utils.db import get_connection

conn = get_connection()
df_all = pd.read_sql(
    "SELECT appid, name_store, genres, release_date, positive, negative FROM steam_indie_list",
    conn
)
conn.close()

df_all["total_reviews"] = df_all["positive"] + df_all["negative"]

# 장르 파싱
def extract_genres(g):
    try:
        return ast.literal_eval(g)
    except:
        return []

df_all["genre_list"] = df_all["genres"].apply(extract_genres)

# 분석 대상 주요 장르 (Indie·Early Access·Free To Play 등 메타 태그 제외)
TARGET_GENRES = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]

for i in TARGET_GENRES:
    # 장르별 리뷰 수 구간(구간 지정은 전체 리뷰 수 기반)
    # 25%ile=6, 50%ile=23, 75%ile=104, 90%ile=562
    bins   = [0, 23, 104, 562, float("inf")]
    labels = ["소형(~23)", "중형(24~104)", "대형(105~562)", "초대형(563+)"]
    df_all["size_tier"] = pd.cut(df_all["total_reviews"], bins=bins, labels=labels)

    print("=== 전체 인디 리뷰 수 분포 ===")
    print(df_all["total_reviews"].describe().round(1))
    print(f"\n구간별 게임 수:\n{df_all['size_tier'].value_counts().sort_index()}")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_6948\1400824912.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_6948\1400824912.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reviews = pd.read_sql(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_6948\1400824912.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_list = pd.read_sql(


=== 전체 인디 리뷰 수 분포 ===
count      61266.0
mean         942.4
std        15098.2
min            0.0
25%            6.0
50%           23.0
75%          104.0
max      1409473.0
Name: total_reviews, dtype: float64

구간별 게임 수:
size_tier
소형(~23)        30640
중형(24~104)     14963
대형(105~562)     9178
초대형(563+)       6116
Name: count, dtype: int64
=== 전체 인디 리뷰 수 분포 ===
count      61266.0
mean         942.4
std        15098.2
min            0.0
25%            6.0
50%           23.0
75%          104.0
max      1409473.0
Name: total_reviews, dtype: float64

구간별 게임 수:
size_tier
소형(~23)        30640
중형(24~104)     14963
대형(105~562)     9178
초대형(563+)       6116
Name: count, dtype: int64
=== 전체 인디 리뷰 수 분포 ===
count      61266.0
mean         942.4
std        15098.2
min            0.0
25%            6.0
50%           23.0
75%          104.0
max      1409473.0
Name: total_reviews, dtype: float64

구간별 게임 수:
size_tier
소형(~23)        30640
중형(24~104)     14963
대형(105~562)     9178
초대형(563+)       6116
Nam

In [8]:
df_clean = df_all.copy()
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], format = 'mixed', errors = 'coerce')
df_2024 = df_clean[(df_clean['release_date'].notna()) & (df_clean['release_date'] >= '2024-01-01') & (df_clean['release_date'] <= '2026-04-20')]
df_2024.sort_values('release_date')

,appid,name_store,genres,release_date,positive,negative,total_reviews,genre_list,size_tier
18305,1993820,T-night,"['Action', 'Adventure', 'Indie']",2024-01-01,7,0,7,"[Action, Adventure, Indie]",소형(~23)
48304,2693810,Galaxy Trek,"['Action', 'Indie', 'Strategy']",2024-01-01,2,0,2,"[Action, Indie, Strategy]",소형(~23)
59877,2703170,Jump Penguin Final,"['Action', 'Casual', 'Indie', 'Early Access']",2024-01-01,1,0,1,"[Action, Casual, Indie, Early Access]",소형(~23)
33128,2723980,Dream Island: A Skyward Journey,"['Action', 'Adventure', 'Indie']",2024-01-01,7,1,8,"[Action, Adventure, Indie]",소형(~23)
33641,2579540,The Fairway Club,"['Adventure', 'Casual', 'Indie', 'Sports']",2024-01-01,25,1,26,"[Adventure, Casual, Indie, Sports]",중형(24~104)
...,...,...,...,...,...,...,...,...,...
52989,2840890,Nekokami - The Human Restoration Project,"['Adventure', 'Casual', 'Indie', 'RPG']",2026-04-13,48,0,48,"[Adventure, Casual, Indie, RPG]",중형(24~104)
45607,1780070,Seeds of Calamity,"['Adventure', 'Casual', 'Indie', 'RPG', 'Simul...",2026-04-13,97,6,103,"[Adventure, Casual, Indie, RPG, Simulation]",중형(24~104)
5866,1431230,Gods of Sand,"['Indie', 'RPG', 'Simulation', 'Strategy']",2026-04-14,560,77,637,"[Indie, RPG, Simulation, Strategy]",초대형(563+)
60045,2861360,Sprite's Honor!,"['Action', 'Adventure', 'Casual', 'Indie', 'Si...",2026-04-15,5,0,5,"[Action, Adventure, Casual, Indie, Simulation]",소형(~23)


In [9]:
df_2024 = df_2024[df_2024['total_reviews'] >= 30]
df_2024

,appid,name_store,genres,release_date,positive,negative,total_reviews,genre_list,size_tier
0,1623730,Palworld,"['Action', 'Adventure', 'Indie', 'RPG', 'Early...",2024-01-18,358266,22443,380709,"[Action, Adventure, Indie, RPG, Early Access]",초대형(563+)
10,899770,Last Epoch,"['Action', 'Adventure', 'Indie', 'RPG']",2024-02-21,88027,22596,110623,"[Action, Adventure, Indie, RPG]",초대형(563+)
20,3164500,Schedule I,"['Action', 'Indie', 'Simulation', 'Strategy', ...",2025-03-24,200803,3238,204041,"[Action, Indie, Simulation, Strategy, Early Ac...",초대형(563+)
23,251570,7 Days to Die,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,327889,42157,370046,"[Action, Adventure, Indie, RPG, Simulation, St...",초대형(563+)
24,1116170,CyberCorp,"['Action', 'Adventure', 'Indie', 'RPG']",2025-04-22,266,56,322,"[Action, Adventure, Indie, RPG]",대형(105~562)
...,...,...,...,...,...,...,...,...,...
60015,2761340,Beats of Betrayal,"['Action', 'Indie', 'RPG', 'Early Access']",2024-08-13,31,0,31,"[Action, Indie, RPG, Early Access]",중형(24~104)
60016,1464170,Touhou Eternal Spell Cards,"['Action', 'Adventure', 'Casual', 'Indie', 'RP...",2024-04-08,30,5,35,"[Action, Adventure, Casual, Indie, RPG, Simula...",중형(24~104)
60113,2336840,Taka Taka,"['Action', 'Adventure', 'Indie', 'Free To Play']",2024-02-29,34,1,35,"[Action, Adventure, Indie, Free To Play]",중형(24~104)
61258,1785940,COVEN,"['Action', 'Adventure', 'Indie', 'Early Access']",2024-10-24,156,8,164,"[Action, Adventure, Indie, Early Access]",대형(105~562)


In [10]:
target_genres = ["Action", "Casual", "Adventure", "Simulation", "Strategy", "RPG", "Racing", "Sports"]
print("=== 장르별 게임 수 ===")
for genre in target_genres:
    cnt = df_2024['genres'].apply(lambda x: genre in x).sum()
    print(f"{genre}: {cnt}개")

=== 장르별 게임 수 ===
Action: 1817개
Casual: 1715개
Adventure: 2139개
Simulation: 1359개
Strategy: 1003개
RPG: 1213개
Racing: 122개
Sports: 145개


In [ ]:
import requests
import time
import datetime

def get_steam_game_review(app_id, start_date):
    
    url = f"https://store.steampowered.com/appreviews/{app_id}"
    
    params = {
        'json': 1,
        'filter': 'recent',
        'language': 'english',
        'review_type': 'all',
        'purchase_type': 'all',
        'num_per_page': 100,
        'cursor' : '*'
    }
    
    reviews = []
    summary = []
    page_count = 1
    end_date = start_date + datetime.timedelta(days = 180)
    
    while True:
        print(f"현재 {app_id}의 리뷰 {page_count}페이지 수집 중... (누적 리뷰: {len(reviews)}개)")
        
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print("API 호출 실패")
            break
            
        data = response.json()
        all_reviews = data.get('reviews', [])
        if not summary:
            summary = data.get('query_summary', [])
        
        if not all_reviews:
            print("더 이상 가져올 리뷰가 없습니다.")
            break
            
        # 리뷰 날짜 확인 루프
        stop_collecting = False
        for review in all_reviews:
            # 리뷰 생성 시점 (Unix Timestamp)
            created_at = datetime.datetime.fromtimestamp(review['timestamp_created'])
            if created_at <= end_date:
                reviews.append(review)
            elif created_at > end_date:
                # 설정한 날짜 이후의 리뷰가 나오면 패스
                continue
            elif created_at < start_date:
                # 설정한 날짜 이전의 리뷰가 나오면 중단 플래그 설정
                stop_collecting = True
                break
        
        if stop_collecting:
            print(f"설정한 날짜({start_date}) 이전 데이터에 도달하여 수집을 종료합니다.")
            break
            
        # 다음 페이지를 위한 커서 갱신
        params['cursor'] = data.get('cursor')
        page_count += 1
        
        # 스팀 서버 부하 방지 및 차단 예방 (중요!)
        time.sleep(1.5)
    
    return reviews, summary
    

In [28]:
genre = 'Casual'
df = df_2024[df_2024['genres'].apply(lambda x: genre in x)]
id_list = df['appid'].to_list()
release_list = df['release_date'].to_list()

df.head()

,appid,name_store,genres,release_date,positive,negative,total_reviews,genre_list,size_tier
96,2379780,Balatro,"['Casual', 'Indie', 'Strategy']",2024-02-20,150524,3042,153566,"[Casual, Indie, Strategy]",초대형(563+)
100,2709570,Supermarket Together,"['Casual', 'Indie', 'Simulation', 'Free To Play']",2024-08-09,63994,3441,67435,"[Casual, Indie, Simulation, Free To Play]",초대형(563+)
127,3097560,Liar's Bar,"['Casual', 'Indie', 'Simulation', 'Strategy', ...",2024-10-02,44341,4209,48550,"[Casual, Indie, Simulation, Strategy, Early Ac...",초대형(563+)
134,2567870,Chained Together,"['Adventure', 'Casual', 'Indie', 'Simulation']",2024-06-19,48180,4958,53138,"[Adventure, Casual, Indie, Simulation]",초대형(563+)
160,2670630,Supermarket Simulator,"['Casual', 'Indie', 'Simulation']",2025-06-19,64104,4237,68341,"[Casual, Indie, Simulation]",초대형(563+)


In [42]:
total_reviews = []
total_summarys = []

for id, release in zip(id_list, release_list):
    reviews, summary = get_steam_game_review(id, release)
    if len(reviews) > 0 and len(summary) > 0:
        total_summarys.extend(summary)
        total_reviews.extend(reviews)
        
        print(f"현재까지 리뷰를 모은 게임 수 {len(total_summarys)}")
        print(f"현재까지 모인 리뷰 수: {len(total_reviews)}")


print("총 리뷰 수", len(total_reviews))
print("총 리뷰 요약 수", len(total_summarys))

현재 2379780의 리뷰 1페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 2페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 3페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 4페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 5페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 6페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 7페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 8페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 9페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 10페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 11페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 12페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 13페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 14페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 15페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 16페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 17페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 18페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 19페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 20페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 21페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 22페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 23페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 24페이지 수집 중... (누적 리뷰: 0개)
현재 2379780의 리뷰 25페이지 수집 중

KeyboardInterrupt: 

In [ ]:
# 총 1시간 43분 경과, 캐주얼 게임 32개의 출시일로부터 6개월(180일)간의 리뷰 355606건 저장
# summary는 아쉽게도 코드를 잘못 짰는지 저장이 잘 안 되었음

df_reviews = pd.DataFrame(total_reviews)
df_summary = pd.DataFrame(total_summarys)

,recommendationid,author,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,...,steam_purchase,received_for_free,refunded,written_during_early_access,primarily_steam_deck,app_release_date,reactions,hardware,timestamp_dev_responded,developer_response
0,172322761,"{'steamid': '76561198148321551', 'personaname'...",english,good,1723906206,1723906206,True,0,0,0.5,...,False,False,False,False,False,1708440403,[],NaN,NaN,NaN
1,172320730,"{'steamid': '76561198018799616', 'personaname'...",english,This is the digital equivalent to crack,1723904174,1723904174,True,0,0,0.5,...,False,False,False,False,False,1708440403,[],NaN,NaN,NaN
2,172320279,"{'steamid': '76561197994140003', 'personaname'...",english,Amazing and simple. I enjoy it a lot :),1723903743,1723903743,True,0,0,0.5,...,True,False,False,False,False,1708440403,[],NaN,NaN,NaN
3,172317078,"{'steamid': '76561198419033950', 'personaname'...",english,good fun,1723900598,1723900598,True,0,0,0.5,...,False,False,False,False,False,1708440403,[],NaN,NaN,NaN
4,172315994,"{'steamid': '76561199544408777', 'personaname'...",english,"ive played 30 hours and hated every second, i ...",1723899579,1723899579,True,0,0,0.5,...,True,False,False,False,True,1708440403,[],NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355601,172166470,"{'steamid': '76561198444145983', 'personaname'...",english,"great game, very adictive",1723720221,1723720221,True,1,0,0.5,...,False,True,False,True,False,1723718829,[],NaN,NaN,NaN
355602,172166226,"{'steamid': '76561198856687955', 'personaname'...",english,"Great game! 10/10 for a factory games veteran,...",1723719902,1723719902,True,10,1,0.504714131355285645,...,False,True,False,True,False,1723718829,"[{'reaction_type': 17, 'count': 1}]",NaN,NaN,NaN
355603,172166104,"{'steamid': '76561198090596389', 'personaname'...",english,Had the privilege to playtest shapez 2 and som...,1723719764,1723719764,True,5,1,0.545454561710357666,...,False,True,False,True,False,1723718829,[],NaN,NaN,NaN
355604,172166100,"{'steamid': '76561199010179978', 'personaname'...",english,"Very good game, pacing is definitely faster th...",1723719758,1723719758,True,1,0,0.5,...,False,False,False,True,False,1723718829,[],NaN,NaN,NaN


In [ ]:
df_reviews.to_csv('casual_reviews_test.csv', index = False)
df_summary.to_csv('casual_r_summary_test.csv', index = False)

In [23]:
import datetime
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].astype('int64')
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].apply(lambda x: datetime.datetime.fromtimestamp(x))

In [25]:
df_reviews['timestamp_created'] = df_reviews['timestamp_created'].dt.tz_localize('UTC').dt.tz_convert('Asia/Seoul')
df_reviews['timestamp_created']

0       2023-03-13 11:48:50+09:00
1       2023-03-29 09:50:29+09:00
2       2023-04-11 11:56:57+09:00
3       2023-05-26 11:07:39+09:00
4       2023-04-17 06:50:31+09:00
                   ...           
13101   2023-12-23 08:39:35+09:00
13102   2023-12-22 15:54:43+09:00
13103   2023-12-22 15:46:45+09:00
13104   2023-12-22 14:14:37+09:00
13105   2023-12-22 11:28:24+09:00
Name: timestamp_created, Length: 13106, dtype: datetime64[us, Asia/Seoul]